# **Initial Phase(by Elsa) **


*   Importation of libraries and Json file
*   Overview of the raw dataset
*   Data cleaning
*   Data preparation





In [ ]:
#importations
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import plotly.graph_objects as go
import json
import requests

# GET request on an endpoint
response = requests. get('https://odre.opendatasoft.com/api/explore/v2.1/catalog/datasets/eco2mix-regional-cons-def/exports/json?lang=fr&timezone=Europe%2FBerlin')
# Display of its content in JSON format
response=response.json()

# Store content in a DataFrame
df_energy=pd.DataFrame(response)

#general info on the dataframe
display(df_energy.shape)
display(df_energy.dtypes)

# overview of the dataset
df_energy.head(10)

In [ ]:
#data cleaning phase - first analyses

#start viewing the missing values
print("nb of columns of the transactions DataFrame containing missing values :",df_energy.isna().any(axis=0).sum(axis=0))
print('total missing values :')
display (df_energy.isna().sum())

#check if there are more missing values on aggregated data than definitive data
#print('missing values for aggregated data :')
#display (df_energy.loc[df_energy['nature']=='Données consolidées'].isna().sum())
#print('missing values for definitive data :')
#display (df_energy.loc[df_energy['nature']=='Données définitives'].isna().sum())

#As column_30 is empty on all rows, we can erase it from our cleaned dataframe (df_energy_cleaned)
df_energy_cleaned=df_energy[['code_insee_region','libelle_region','nature','date','heure','date_heure','consommation','thermique','nucleaire','eolien','solaire','hydraulique','pompage','bioenergies','ech_physiques', 'stockage_batterie', 'destockage_batterie', 'eolien_terrestre','eolien_offshore','tco_thermique','tch_thermique','tco_nucleaire','tch_nucleaire','tco_eolien','tch_eolien','tco_solaire','tch_solaire','tco_hydraulique','tch_hydraulique','tco_bioenergies','tch_bioenergies']]
print ("shape of the target dataframe=",df_energy_cleaned.shape)

#check negative values
cols = ['consommation','thermique','nucleaire','solaire','hydraulique','pompage','bioenergies','ech_physiques', 'stockage_batterie', 'destockage_batterie', 'eolien_terrestre','eolien_offshore','tco_thermique','tch_thermique','tco_nucleaire','tch_nucleaire','tco_eolien','tch_eolien','tco_solaire','tch_solaire','tco_hydraulique','tch_hydraulique','tco_bioenergies','tch_bioenergies']
for col in cols:
    print(f"negative values '{col}':",
          (df_energy_cleaned[col] < 0).sum())


In [ ]:
#data cleaning phase - Corrections of values and formats
#set all numerical missing values to 0
df_energy_cleaned=df_energy_cleaned.fillna(0)

#convert to the proper type of data columns date, heure and date_heure
df_energy_cleaned['date_heure']=df_energy_cleaned['date_heure'].astype('datetime64[ns, UTC]')
df_energy_cleaned['date']=pd.to_datetime( df_energy_cleaned['date'],format='ISO8601')
df_energy_cleaned['heure']== pd.to_datetime(df_energy_cleaned["heure"], format="%H:%M").dt.time

# remove anomalies from eolien
df_energy_cleaned=df_energy_cleaned.replace({'eolien': 'nan'}, 0)
df_energy_cleaned=df_energy_cleaned.replace({'eolien': 'ND'}, 0)
df_energy_cleaned=df_energy_cleaned.replace({'eolien': '-'}, 0)
df_energy_cleaned['eolien'] = df_energy_cleaned['eolien'].astype ("Float64")
#print("negative values 'eolien':",df_energy_cleaned['eolien'].loc[df_energy_cleaned['eolien']<0].count())
#print('')
df_energy_cleaned.loc[df_energy_cleaned['eolien'] < 0, 'eolien'] = 0

# replace negative values by 0 for columns 'thermique',Nucleaire, 'solaire','hydraulique','pompage', 'ech_physiques'
df_energy_cleaned.loc[df_energy_cleaned['thermique'] < 0, 'thermique'] = 0
cols = ['solaire', 'hydraulique','pompage', 'ech_physiques','nucleaire']
df_energy_cleaned[cols] = df_energy_cleaned[cols].clip(lower=0)
#for col in cols:
    #print(f"negative values '{col}':",
          #(df_energy_cleaned[col] < 0).sum())

print('Cleaning phase is finished.')

In [ ]:
#Data preparation  - check data quality

#list values of qualitative attributes
display (df_energy_cleaned['code_insee_region'].unique())
display (df_energy_cleaned['libelle_region'].unique())
display (df_energy_cleaned['nature'].unique())

#check distribution of quantitative attributes
display(df_energy_cleaned.describe())

Related Data Audit file is available here :

```
# https://docs.google.com/spreadsheets/d/1Sm4n9Jx5DVdKHfUIZrq4sKB0bMYWmoF6naFtgWgiXU4/edit?usp=sharing
```



In [ ]:
df_energy_cleaned.head()


# **Analysis phase (by Pascal)**

In [ ]:
# REGIONAL STATISTICAL ANALYSIS
# Group the dataset by region
# dataset has a panel structure (region x time)
# compute statistics per region instead of global aggregates


group_cols = ["code_insee_region", "libelle_region"]

# I select the main quantitative energy variables.
# These represent production and consumption levels in MW.
energy_cols = [
    "consommation",
    "thermique",
    "nucleaire",
    "eolien",
    "solaire",
    "hydraulique",
    "pompage",
    "bioenergies",
    "ech_physiques"
]

# Keep only variables that actually exist in the dataset
energy_cols = [col for col in energy_cols if col in df_energy_cleaned.columns]

print("Energy variables used for analysis:", energy_cols)

# Non-convertible values will be transformed into NaN.

df_tmp = df_energy_cleaned.copy()

for col in energy_cols:
    df_tmp[col] = pd.to_numeric(df_tmp[col], errors="coerce")

# Region codes should also be numeric for proper sorting
df_tmp["code_insee_region"] = pd.to_numeric(
    df_tmp["code_insee_region"], errors="coerce"
)

# I aggregate the data by region and compute:
# - mean → average production/consumption level
# - std  → volatility over time
# - min  → lowest observed value
# - max  → peak value

regional_stats = (
    df_tmp
    .groupby(group_cols)[energy_cols]
    .agg(["mean", "std", "min", "max"])
)

# Flatten the multi-level column names for readability
regional_stats.columns = [
    f"{variable}_{stat}" for variable, stat in regional_stats.columns
]

# Convert index back to regular columns and sort by region code
regional_stats = (
    regional_stats
    .reset_index()
    .sort_values("code_insee_region")
)

# Display final table
display(regional_stats)


The regional statistics reveal substantial heterogeneity in electricity consumption across French regions.

Île-de-France and Auvergne-Rhône-Alpes exhibit the highest average consumption levels, reflecting their economic and demographic importance.

Smaller regions such as Centre-Val de Loire and Bretagne display significantly lower average demand, consistent with lower population density.

The standard deviation of consumption scales with region size, indicating that larger regions experience stronger absolute fluctuations.

However, volatility should ideally be evaluated relative to the mean (coefficient of variation), as absolute standard deviation alone may be misleading.

Thermal production shows strong regional concentration, particularly in Grand Est and Hauts-de-France, suggesting structural differences in generation infrastructure.

Nuclear production is unevenly distributed, reflecting the geographic placement of nuclear power plants.

Wind production (eolien) demonstrates higher volatility compared to nuclear generation, highlighting the intermittency of renewable sources.

Solar production remains relatively modest in absolute terms but likely exhibits strong intra-day and seasonal variability.

Hydroelectric production appears more stable but varies depending on regional natural resources and reservoir capacity.

Several regions show significant negative values in physical exchanges (ech_physiques), indicating net electricity exports.

In contrast, Île-de-France exhibits large positive exchange values, suggesting structural dependence on imports from other regions.

Pumped storage (pompage) activity is limited to specific regions, indicating localized storage infrastructure.

The energy mix differs substantially across regions, underlining the importance of region-level analysis rather than global aggregation.

Overall, the dataset confirms that regional electricity systems differ in scale, volatility, and structural composition, which has implications for grid stability and energy policy analysis.

# **Initial Data Exploration (by Ahmed)**

In [ ]:
#
# --- Final Corrected Code - Version 4 ---

# 1. Make a fresh, safe copy of the cleaned data
data = df_energy_cleaned.copy()

# 2. Correct the 'eolien' column
data['eolien'] = data['eolien'].astype(str).str.replace(',', '.', regex=False)
data['eolien'] = pd.to_numeric(data['eolien'], errors='coerce')

# 3. Aggregate Consumption at the national level
# We assume all rows can potentially describe consumption
national_consumption = data.groupby('date_heure')['consommation'].sum().reset_index()

# 4. Aggregate Production at the national level
# We assume the SAME rows can also describe production
production_cols = ['thermique', 'nucleaire', 'eolien', 'solaire', 'hydraulique', 'bioenergies']
data['total_production'] = data[production_cols].fillna(0).sum(axis=1)
national_production = data.groupby('date_heure')['total_production'].sum().reset_index()

# 5. Merge the national data
df_national = pd.merge(national_consumption, national_production, on='date_heure')

# --- Final Inspection ---
print("--- Final National DataFrame ---")
print("Shape:", df_national.shape)
print("\nFirst 5 rows:")
display(df_national.head())
print("\nStatistics:")
display(df_national.describe())

In [ ]:
# --- National Consumption vs. Production (Enhanced Version) ---

# 1. Aggregate data to a national, daily level (same as before)
df_national_temp = df_energy_cleaned.copy()
df_national_temp['date'] = pd.to_datetime(df_national_temp['date_heure']).dt.date
production_cols = ['thermique', 'nucleaire', 'eolien', 'solaire', 'hydraulique', 'bioenergies']
df_national_temp['total_production'] = df_national_temp[production_cols].sum(axis=1)

df_national_simple = df_national_temp.groupby('date')[['consommation', 'total_production']].mean().reset_index()

# 2. Create the plot
fig_nat_enhanced = go.Figure()

# 3. --- Add the shaded area between the lines (the key improvement) ---
#    We draw the production line, then the consumption line in reverse,
#    and use 'fill="toself"' to shade the area enclosed.
fig_nat_enhanced.add_trace(go.Scatter(
    x=df_national_simple['date'],
    y=df_national_simple['total_production'],
    mode='lines',
    line=dict(width=0), # Make the line itself invisible
    showlegend=False # Don't show this helper trace in the legend
))
fig_nat_enhanced.add_trace(go.Scatter(
    x=list(df_national_simple['date'])[::-1], # Reversed date list
    y=list(df_national_simple['consommation'])[::-1], # Reversed consumption list
    mode='lines',
    line=dict(width=0),
    fill='toself', # This is the magic that creates the shaded area
    fillcolor='rgba(0,100,80,0.2)', # A semi-transparent green color
    name='Surplus/Deficit',
    showlegend=False
))


# 4. Add the actual visible lines for Production and Consumption on top of the shaded area
fig_nat_enhanced.add_trace(go.Scatter(
    x=df_national_simple['date'],
    y=df_national_simple['total_production'],
    mode='lines',
    name='National Production',
    line=dict(color='green', width=2)
))
fig_nat_enhanced.add_trace(go.Scatter(
    x=df_national_simple['date'],
    y=df_national_simple['consommation'],
    mode='lines',
    name='National Consumption',
    line=dict(color='royalblue', width=2)
))


# 5. Update the layout with a title and the range slider
fig_nat_enhanced.update_layout(
    title_text='National Energy Balance: Production vs. Consumption',
    xaxis_title='Date',
    yaxis_title='Average Daily Energy (MW)',
    # Add the range slider
    xaxis_rangeslider_visible=True
)

fig_nat_enhanced.show()


Figure 1: National Energy Consumption vs. Production (2013-2024) . This chart illustrates the national energy balance over the last decade. A clear seasonal pattern is visible, with both consumption and production peaking during the winter months. Notably, national production consistently exceeds consumption, indicating a surplus that is likely exported.

4. Regional Level Analysis: Consumption vs. Production (By Ahmed)

In [ ]:
#
# --- Prepare Data for Regional Analysis ---

# We will use the 'data' DataFrame which is already cleaned.
# We just need to group it by region AND timestamp.

# Group by region and date to get consumption for each region over time.
regional_consumption = data.groupby(['libelle_region', 'date_heure'])['consommation'].sum().reset_index()

# Group by region and date to get total production for each region over time.
regional_production = data.groupby(['libelle_region', 'date_heure'])['total_production'].sum().reset_index()

# Merge the two dataframes to have everything in one place.
df_regional = pd.merge(regional_consumption, regional_production, on=['libelle_region', 'date_heure'])

print("Regional data prepared successfully.")
display(df_regional.head())

In [ ]:
# --- Detailed National Energy Mix (Daily Average) ---

# Aggregate data to a national, daily level with all energy types
df_national_detailed_temp = df_energy_cleaned.copy()
df_national_detailed_temp['date'] = pd.to_datetime(df_national_detailed_temp['date_heure']).dt.date
energy_cols = ['consommation', 'thermique', 'nucleaire', 'eolien', 'solaire', 'hydraulique', 'bioenergies']

# Group by date only for national totals
df_national_detailed = df_national_detailed_temp.groupby('date')[energy_cols].mean().reset_index()

# Create the plot
fig_nat_2 = go.Figure()

# Stacked traces
fig_nat_2.add_trace(go.Scatter(x=df_national_detailed['date'], y=df_national_detailed['hydraulique'], name='Hydro', stackgroup='one', line_width=0, fillcolor='dodgerblue'))
fig_nat_2.add_trace(go.Scatter(x=df_national_detailed['date'], y=df_national_detailed['eolien'], name='Wind', stackgroup='one', line_width=0, fillcolor='lightgreen'))
fig_nat_2.add_trace(go.Scatter(x=df_national_detailed['date'], y=df_national_detailed['solaire'], name='Solar', stackgroup='one', line_width=0, fillcolor='gold'))
fig_nat_2.add_trace(go.Scatter(x=df_national_detailed['date'], y=df_national_detailed['thermique'], name='Thermal', stackgroup='one', line_width=0, fillcolor='darkred'))
fig_nat_2.add_trace(go.Scatter(x=df_national_detailed['date'], y=df_national_detailed['bioenergies'], name='Bioenergy', stackgroup='one', line_width=0, fillcolor='saddlebrown'))

# Separate lines
fig_nat_2.add_trace(go.Scatter(x=df_national_detailed['date'], y=df_national_detailed['nucleaire'], name='Nuclear', line=dict(color='orange', width=2.5)))
fig_nat_2.add_trace(go.Scatter(x=df_national_detailed['date'], y=df_national_detailed['consommation'], name='Consumption', line=dict(color='black', width=3)))

fig_nat_2.update_layout(
    title_text='Detailed National Energy Mix (Nuclear Separated)',
    xaxis_title='Date',
    yaxis_title='Average Daily Energy (MW)'
)

fig_nat_2.show()


Figure 2: Interactive Analysis of Regional Energy Balance. This interactive tool allows for a detailed examination of the energy balance within each French region. By selecting a region from the dropdown menu, we can observe specific local patterns. For example, regions with high industrial activity show different consumption profiles compared to more residential areas."""

"Analysis by production sector: nuclear/renewable energy"

*   Élément de liste

*   Élément de liste
*   Élément de liste


In [ ]:
# --- Detailed National Energy Mix (Enhanced Version) ---

# 1. Aggregate data to a national, daily level (the key to preventing freezes)
df_nat_detailed_temp = df_energy_cleaned.copy()
df_nat_detailed_temp['date'] = pd.to_datetime(df_nat_detailed_temp['date_heure']).dt.date
energy_cols = ['consommation', 'thermique', 'nucleaire', 'eolien', 'solaire', 'hydraulique', 'bioenergies']

# Group by date to get the national daily average for each energy type
df_national_detailed = df_nat_detailed_temp.groupby('date')[energy_cols].mean().reset_index()

# 2. Create the plot object
fig_nat_detailed = go.Figure()

# 3. --- Add the STACKED traces for non-nuclear production ---
# This makes the details of smaller sources visible.
fig_nat_detailed.add_trace(go.Scatter(
    x=df_national_detailed['date'], y=df_national_detailed['hydraulique'], name='Hydro',
    mode='lines', stackgroup='one', line_width=0, fillcolor='dodgerblue'
))
fig_nat_detailed.add_trace(go.Scatter(
    x=df_national_detailed['date'], y=df_national_detailed['eolien'], name='Wind',
    mode='lines', stackgroup='one', line_width=0, fillcolor='lightgreen'
))
fig_nat_detailed.add_trace(go.Scatter(
    x=df_national_detailed['date'], y=df_national_detailed['solaire'], name='Solar',
    mode='lines', stackgroup='one', line_width=0, fillcolor='gold'
))
fig_nat_detailed.add_trace(go.Scatter(
    x=df_national_detailed['date'], y=df_national_detailed['thermique'], name='Thermal',
    mode='lines', stackgroup='one', line_width=0, fillcolor='darkred'
))
fig_nat_detailed.add_trace(go.Scatter(
    x=df_national_detailed['date'], y=df_national_detailed['bioenergies'], name='Bioenergy',
    mode='lines', stackgroup='one', line_width=0, fillcolor='saddlebrown'
))

# 4. --- Add SEPARATE, clear lines for Nuclear and Consumption ---
fig_nat_detailed.add_trace(go.Scatter(
    x=df_national_detailed['date'], y=df_national_detailed['nucleaire'], name='Nuclear',
    mode='lines', line=dict(color='orange', width=2.5)
))
fig_nat_detailed.add_trace(go.Scatter(
    x=df_national_detailed['date'], y=df_national_detailed['consommation'], name='Consumption',
    mode='lines', line=dict(color='black', width=3, dash='dash') # Dashed line to differentiate it
))

# 5. Update the layout with a title and the range slider
fig_nat_detailed.update_layout(
    title_text='Composition of National Energy Production by Sector',
    xaxis_title='Date',
    yaxis_title='Average Daily Energy (MW)',
    legend_title_text='Energy Source',
    # Add the range slider for better interactivity
    xaxis_rangeslider_visible=True
)

fig_nat_detailed.show()


Figure 3: Composition of National Energy Production by Sector.                                                           The stacked area chart reveals the composition of France's energy mix. Nuclear power (orange) forms the vast and stable baseload of production. We can also observe the seasonal contribution of hydraulic power (blue) and the smaller but growing shares of solar (gold) and wind (green) energy over the years.

In [ ]:
# --- Interactive Geographical Distribution of Renewable Energy ---

# 1. Prepare Data for the Map
renewable_cols = ['eolien', 'solaire', 'hydraulique', 'bioenergies']

# Group by region and get the total production for each renewable type.
# We use sum() here because we are looking at the total accumulated production over the period.
renewable_by_region = df_energy_cleaned.groupby('libelle_region')[renewable_cols].sum().reset_index()

# Calculate the total renewable production as a new column
renewable_by_region['total_renewable'] = renewable_by_region[renewable_cols].sum(axis=1)

# 2. Load GeoJSON data for French regions (same as before)
geojson_url = "https://raw.githubusercontent.com/gregoiredavid/france-geojson/master/regions.geojson"
try:
    response = requests.get(geojson_url )
    geojson_regions = response.json()
except Exception as e:
    print(f"Failed to load GeoJSON file: {e}")
    geojson_regions = None # Set to None if it fails

# 3. Create the Interactive Choropleth Map
if geojson_regions: # Only proceed if the GeoJSON was loaded successfully
    fig_map = go.Figure()

    # Define the columns the user can select from the dropdown
    columns_to_plot = ['total_renewable', 'hydraulique', 'eolien', 'solaire', 'bioenergies']

    # --- Create one map trace for each energy type, but make them invisible ---
    for col in columns_to_plot:
        fig_map.add_trace(
            go.Choropleth(
                geojson=geojson_regions,
                locations=renewable_by_region['libelle_region'],
                featureidkey="properties.nom",
                z=renewable_by_region[col], # 'z' is the value that determines the color
                colorscale="Viridis",
                colorbar_title="GWh", # Generic title for the color bar
                name=col.capitalize(),
                visible=False # Make all traces invisible initially
            )
        )

    # --- Make the first trace (Total Renewable) visible by default ---
    fig_map.data[0].visible = True

    # 4. --- Create the Dropdown Menu ---
    buttons = []
    for i, col in enumerate(columns_to_plot):
        # Create a visibility mask. It's a list of True/False values.
        visibility = [False] * len(columns_to_plot)
        visibility[i] = True # Only the selected trace is visible

        button = dict(
            label=col.replace('_', ' ').capitalize(), # e.g., 'total_renewable' -> 'Total renewable'
            method="update",
            args=[{"visible": visibility}]
        )
        buttons.append(button)

    # 5. Update the layout to add the dropdown menu
    fig_map.update_layout(
        title_text='Geographical Distribution of Renewable Energy Production',
        title_x=0.5,
        updatemenus=[dict(
            active=0,
            buttons=buttons,
            direction="down",
            x=0.05, xanchor="left",
            y=0.95, yanchor="top"
        )],
        geo=dict(
            scope='europe',
            fitbounds="locations", # Zoom to France
            visible=False
        )
    )

    fig_map.show()


Figure 4: Geographical Distribution of Total Renewable Energy Production. This choropleth map answers the question of "where" renewable energy is located. It highlights that production is not evenly distributed. The Auvergne-Rhône-Alpes region is the dominant producer, largely due to its significant hydraulic resources. Other regions like Occitanie also show substantial renewable output, indicating strong wind and solar capacities.

In [ ]:
#Next cell